<a href="https://colab.research.google.com/github/specter-26/ta2_reddit_project/blob/main/notebooks/01_data_collection_and_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01. Data Collection, Merging & Preprocessing Pipeline

**Project:** Text Analytics 2 — Analyzing Dating Problems and Advice on Reddit  
**Deliverable:** Reproducible data ingestion, deduplication, noise-filtering, and anonymization pipeline producing paired parent posts and top-level comments (`parentPostId` preserved).

> **Data Privacy & Compliance Note:** Per project guidelines and Reddit's Terms of Service, raw scraped JSON files and full datasets are excluded from this public repository (stored privately for team use). This notebook documents the exact collection schema, cleaning logic, quality assertions, exploratory statistics, and generates the small anonymized sample stored in `data/sample/`.

### Pipeline Overview
1. **Load & Inspect Raw Scrapes:** Load the 8 Apify JSON exports (`coding-doctor-omar/reddit-scraper-pro`) from `raw_data/` and verify schema consistency across all files.
2. **Merge, Deduplicate & Clean:**
   - Combine records across `r/dating_advice`, `r/relationship_advice`, `r/datingadvice`, and `r/relationshipadvice`.
   - Split records into separate `posts` and `comments` tables using `submissionType`.
   - Retain **top-level comments only** (`depth == 0`) so nested reply chains do not distort advice frequency counts.
   - Deduplicate posts and comments by `submissionId` (retaining the highest `score` when threads appeared across multiple sort filters).
   - Remove `[deleted]`, `[removed]`, `[unavailable]`, empty texts, and `AutoModerator` entries.
   - Remove cross-subreddit duplicate post texts (`bodyText`) to prevent train/test data leakage, and drop posts with zero surviving comments so every retained thread contains paired advice.
3. **Data Quality Verification:** Run automated assertions and bot/boilerplate audits to verify that zero duplicates, subcomments, orphan comments, or moderator notices remain.
4. **Exploratory Data Analysis (EDA):** Inspect subreddit distributions, multi-year temporal coverage, comment density per post, text lengths, and metadata coverage.
5. **Anonymize & Export Datasets:** Hash usernames (`author_anon`) and export full compressed datasets (`.csv.gz`) for private team modeling alongside a 50-thread anonymized sample for public GitHub inspection.
6. **Post-Export Verification:** Reload exported files from disk to verify integrity and inspect a sample paired thread.

## 1. Load & Inspect Raw Apify JSON Files

We load all 8 raw `.json` exports from the local `raw_data/` folder (shared privately among team members) and audit their record counts, submission types, and key schemas to confirm structural consistency before merging.

In [9]:
import glob
import json
import os
from collections import Counter
import pandas as pd

# --- 1. VERIFY LOCAL RAW DATA FOLDER ---
files = sorted(glob.glob("raw_data/*.json"))
assert len(files) > 0, (
    "No JSON files found in 'raw_data/'. "
    "Please place the 8 raw Apify JSON files into 'raw_data/' before running."
)
print(f"Found {len(files)} raw JSON files in 'raw_data/':\n")

# --- 2. AUDIT PER-FILE COUNTS & SCHEMA CONSISTENCY ---
summary = []
unique_key_sets = set()

for fp in files:
    fname = os.path.basename(fp)
    with open(fp, "r", encoding="utf-8") as f:
        data = json.load(f)

    key_patterns = {tuple(sorted(item.keys())) for item in data}
    unique_key_sets.update(key_patterns)

    types = Counter(item.get("submissionType", "unknown") for item in data)
    subreddits = Counter(item.get("subreddit", "unknown") for item in data)

    summary.append({
        "file": fname,
        "total_records": len(data),
        "unique_schemas": len(key_patterns),
        "record_types": dict(types),
        "subreddits": dict(subreddits),
    })

print(f"Distinct key schemas across all {len(files)} files: {len(unique_key_sets)}")
summary_df = pd.DataFrame(summary)
pd.set_option("display.max_colwidth", None)
display(summary_df)

Found 8 raw JSON files in 'raw_data/':

Distinct key schemas across all 8 files: 1


,file,total_records,unique_schemas,record_types,subreddits
0,dataset_Reddit-Scraper-Pro_2026-11-10_19-48-45-184.json,8439,1,"{'post': 901, 'comment': 7538}","{'r/relationshipadvice': 5813, 'r/datingadvice': 2626}"
1,dataset_Reddit-Scraper-Pro_2026-11-12_13-18-12-181.json,3843,1,"{'post': 531, 'comment': 3312}","{'r/relationshipadvice': 1611, 'r/datingadvice': 2232}"
2,dataset_Reddit-Scraper-Pro_2026-11-12_13-38-49-052.json,16719,1,"{'post': 767, 'comment': 15952}","{'r/datingadvice': 7274, 'r/relationshipadvice': 9445}"
3,dataset_Reddit-Scraper-Pro_2026-11-12_13-44-18-220.json,14076,1,"{'post': 784, 'comment': 13292}","{'r/relationshipadvice': 7763, 'r/datingadvice': 6313}"
4,dataset_Reddit-Scraper-Pro_2026-11-13_15-32-29-894.json,16000,1,"{'post': 226, 'comment': 15774}","{'r/dating_advice': 10382, 'r/relationship_advice': 5618}"
5,dataset_Reddit-Scraper-Pro_2026-11-13_16-12-10-413.json,13800,1,"{'post': 131, 'comment': 13669}","{'r/dating_advice': 5918, 'r/relationship_advice': 7882}"
6,dataset_Reddit-Scraper-Pro_2026-11-13_17-07-46-777.json,18150,1,"{'post': 179, 'comment': 17971}","{'r/dating_advice': 11699, 'r/relationship_advice': 6451}"
7,dataset_Reddit-Scraper-Pro_2026-11-13_20-03-31-080.json,15100,1,"{'post': 127, 'comment': 14973}",{'r/dating_advice': 15100}


### Raw Dataset Schema Reference (24 Fields)

All 8 raw JSON files share the exact same 24-field flat structure, where posts and comments are stored as separate records linked via `submissionId` and `parentPostId`:

#### Identifiers & Thread Hierarchy
| Key | Type | Description |
| :--- | :--- | :--- |
| `submissionId` | `string` | Unique Reddit ID for the post or comment (used for deduplication). |
| `submissionIdPrefixed` | `string` | Reddit fullname with type prefix (`t3_` for posts, `t1_` for comments). |
| `submissionType` | `string` | Record type: `"post"` or `"comment"`. |
| `parentId` | `string \| null` | ID of the direct parent (`null` for posts and top-level comments; parent comment ID for nested replies). |
| `parentPostId` | `string \| null` | Prefixed ID (`t3_...`) of the root parent post (`null` for posts; links every comment back to its thread). |
| `depth` | `int \| null` | Comment nesting level (`null` for posts; `0` for top-level comments; `1+` for nested replies). |

#### Core Text & Content
| Key | Type | Description |
| :--- | :--- | :--- |
| `title` | `string \| null` | Title of the Reddit post (`null` for comments). |
| `bodyText` | `string \| null` | Main body text of the post or comment (may contain `[deleted]` or `[removed]`). |
| `author` | `string` | Reddit username of the author (e.g., `AutoModerator` or `[deleted]`). |
| `subreddit` | `string` | Subreddit name (`r/dating_advice`, `r/relationship_advice`, `r/datingadvice`, `r/relationshipadvice`). |
| `submittedAt` | `string` | Creation timestamp in UTC ISO-8601 format. |
| `postType` | `string \| null` | Format of the post (`text`, `link`, `image`, `gallery`, `video`, `poll`, or `crosspost`; `null` for comments). |

#### Engagement & Metadata
| Key | Type | Description |
| :--- | :--- | :--- |
| `score` | `int` | Net upvote score (upvotes minus downvotes) at the time of scraping. |
| `upvoteRatio` | `float \| null` | Ratio of upvotes to total votes on a post (`0.0`–`1.0`; `null` for comments). |
| `numComments` | `int \| null` | Total comment count on the post at scrape time (`null` for comments). |
| `postFlair` | `string \| null` | Subreddit category flair attached to the post (`null` for comments). |
| `tags` | `list` | Post tags such as `Spoiler` or `NSFW` (empty list `[]` if none). |

#### URLs, Crossposts & Media Links
| Key | Type | Description |
| :--- | :--- | :--- |
| `submissionUrl` | `string` | Direct Reddit URL to the post or comment. |
| `parentPostUrl` | `string \| null` | Direct URL to the parent post (`null` for posts). |
| `crosspostParentId` | `string \| null` | Original post ID if the submission is a crosspost (`null` otherwise). |
| `crosspostParentUrl` | `string \| null` | Original post URL if the submission is a crosspost (`null` otherwise). |
| `externalUrl` | `string \| null` | External link URL for link-type posts (`null` for text posts/comments). |
| `thumbnailUrl` | `string \| null` | Thumbnail preview image URL (`null` if none). |
| `galleryImagesUrls` | `list` | List of image URLs for gallery posts (empty list `[]` otherwise). |

## 2. Merge, Deduplicate & Filter Noise

Here we combine all 8 raw JSON files and apply a sequential cleaning funnel:
1. **Split by `submissionType`:** Separate parent posts (`"post"`) from comments (`"comment"`).
2. **Exclude subcomments:** Filter comments strictly to `depth == 0` so only direct top-level replies to the parent post are retained.
3. **Deduplicate by ID:** Because scrapes overlapped across sort filters (`default`, `top-past-year`, `top-past-month`, `top-all-time`), drop duplicate `submissionId` entries while keeping the record with the highest `score`.
4. **Filter deleted & automated content:** Remove rows where `title` or `bodyText` is empty, `[deleted]`, `[removed]`, or `[unavailable]`, as well as rows authored by `AutoModerator` or `[deleted]`.
5. **Standardize IDs & align parent-child threads:**
   - Strip the `t3_` prefix from `comments["parentPostId"]` so it matches `posts["submissionId"]` cleanly.
   - Drop duplicate cross-posted `bodyText` entries among posts to prevent train/test split leakage.
   - Retain only posts that have at least one surviving clean top-level comment, and remove any orphan comments.

In [10]:
# --- 1. MERGE ALL RAW JSON FILES ---
all_records = []
for fp in files:
    fname = os.path.basename(fp)
    with open(fp, "r", encoding="utf-8") as f:
        records = json.load(f)
    for r in records:
        r["source_file"] = fname
    all_records.extend(records)

df_raw = pd.DataFrame(all_records)
df_posts_raw = df_raw[df_raw["submissionType"] == "post"].copy()
df_comments_raw = df_raw[df_raw["submissionType"] == "comment"].copy()

print(f"Total merged raw records: {len(df_raw):,} ({len(df_posts_raw):,} posts, {len(df_comments_raw):,} comments)\n")

def is_valid_text(series):
    cleaned = series.fillna("").astype(str).str.strip()
    return (cleaned != "") & (~cleaned.str.lower().isin(["[deleted]", "[removed]", "[unavailable]"]))

# --- 2. CLEAN POSTS ---
posts = df_posts_raw.copy()
n_posts_start = len(posts)

# Deduplicate by submissionId (keep highest score if scraped multiple times)
posts = posts.sort_values("score", ascending=False).drop_duplicates(subset=["submissionId"], keep="first")
n_posts_dedup = len(posts)

# Remove deleted/removed/empty posts & AutoModerator
posts = posts[
    is_valid_text(posts["title"]) &
    is_valid_text(posts["bodyText"]) &
    (~posts["author"].fillna("").str.lower().isin(["automoderator", "[deleted]", "[removed]"]))
].copy()
n_posts_textclean = len(posts)

# Remove identical bodyText cross-posted across subreddits (prevents train/test leakage)
posts = posts.drop_duplicates(subset=["bodyText"], keep="first").copy()
n_posts_unique_text = len(posts)

# --- 3. CLEAN COMMENTS ---
comments = df_comments_raw.copy()
n_comm_start = len(comments)

# Keep ONLY top-level comments (depth == 0; excludes all nested subcomments depth >= 1)
comments = comments[
    (comments["depth"] == 0) &
    (comments["parentPostId"].notna())
].copy()
n_comm_toplevel = len(comments)

# Deduplicate by submissionId (keep highest score if scraped multiple times)
comments = comments.sort_values("score", ascending=False).drop_duplicates(subset=["submissionId"], keep="first")
n_comm_dedup = len(comments)

# Remove deleted/removed/empty comments & AutoModerator
comments = comments[
    is_valid_text(comments["bodyText"]) &
    (~comments["author"].fillna("").str.lower().isin(["automoderator", "[deleted]", "[removed]"]))
].copy()
n_comm_textclean = len(comments)

# --- 4. STANDARDIZE THREAD IDS & ALIGN POSTS WITH COMMENTS ---
# Strip 't3_' prefix from parentPostId so it matches posts['submissionId']
comments["parentPostId"] = comments["parentPostId"].astype(str).str.replace("^t3_", "", regex=True)

# Keep only comments whose parent post survived cleaning
comments = comments[comments["parentPostId"].isin(set(posts["submissionId"].astype(str)))].copy()
n_comm_linked = len(comments)

# Keep only posts that have >= 1 surviving clean top-level comment
posts = posts[posts["submissionId"].isin(set(comments["parentPostId"]))].copy()
n_posts_final = len(posts)

# Re-sync comments to guarantee zero orphan comments
comments = comments[comments["parentPostId"].isin(set(posts["submissionId"]))].copy()
n_comm_final = len(comments)

print("=== CLEANING FUNNEL SUMMARY ===")
print(
    f"Posts:    {n_posts_start:,} raw -> {n_posts_dedup:,} ID-dedup -> "
    f"{n_posts_textclean:,} text-clean -> {n_posts_unique_text:,} unique-text -> "
    f"{n_posts_final:,} paired (>=1 comment)"
)
print(
    f"Comments: {n_comm_start:,} raw -> {n_comm_toplevel:,} top-level -> "
    f"{n_comm_dedup:,} ID-dedup -> {n_comm_textclean:,} text-clean -> "
    f"{n_comm_final:,} linked (uncapped)"
)

Total merged raw records: 106,127 (3,646 posts, 102,481 comments)

=== CLEANING FUNNEL SUMMARY ===
Posts:    3,646 raw -> 2,609 ID-dedup -> 2,606 text-clean -> 2,603 unique-text -> 2,341 paired (>=1 comment)
Comments: 102,481 raw -> 32,982 top-level -> 23,650 ID-dedup -> 21,095 text-clean -> 20,076 linked (uncapped)


## 3. Data Quality Verification

Before running exploratory analysis or exporting files, we run automated assertions and a bot/boilerplate audit to guarantee that zero duplicate IDs, nested subcomments, `[deleted]` placeholders, `AutoModerator` entries, or unlinked comments remain.

In [11]:
import hashlib

# --- 1. AUTOMATED ASSERTIONS ---
bad_authors = {"automoderator", "[deleted]", "[removed]"}
bad_texts = {"", "[deleted]", "[removed]", "[unavailable]"}

assert not posts["author"].fillna("").str.lower().isin(bad_authors).any(), "Found bad author in posts!"
assert not comments["author"].fillna("").str.lower().isin(bad_authors).any(), "Found bad author in comments!"
assert not posts["title"].fillna("").str.strip().str.lower().isin(bad_texts).any(), "Found deleted title!"
assert not posts["bodyText"].fillna("").str.strip().str.lower().isin(bad_texts).any(), "Found deleted post body!"
assert not comments["bodyText"].fillna("").str.strip().str.lower().isin(bad_texts).any(), "Found deleted comment body!"
assert (comments["depth"] == 0).all(), "Found subcomments with depth != 0!"
assert posts["submissionId"].is_unique, "Duplicate post IDs remain!"
assert comments["submissionId"].is_unique, "Duplicate comment IDs remain!"
assert posts["bodyText"].is_unique, "Duplicate post bodyText remains!"
assert comments["parentPostId"].isin(set(posts["submissionId"])).all(), "Orphan comments remain!"
assert posts["submissionId"].isin(set(comments["parentPostId"])).all(), "Comment-less posts remain!"

print("ALL ASSERTIONS PASSED: 0 AutoModerator, 0 [deleted]/[removed], 0 subcomments, 0 duplicates, 100% paired threads.\n")

# --- 2. ANONYMIZE & RUN BOT / BOILERPLATE AUDIT ---
SALT = "ta2_reddit_2026"
posts["author_anon"] = posts["author"].apply(
    lambda x: "user_" + hashlib.sha256(f"{SALT}_{x}".encode()).hexdigest()[:10]
)
comments["author_anon"] = comments["author"].apply(
    lambda x: "user_" + hashlib.sha256(f"{SALT}_{x}".encode()).hexdigest()[:10]
)

print("=== TOP 10 COMMENT AUTHORS (ANONYMIZED BOT CHECK) ===")
print(comments["author_anon"].value_counts().head(10))

mod_phrases = r"your post has been removed|this post has been locked|please contact the moderators|i am a bot"
bot_text_hits = comments[comments["bodyText"].str.contains(mod_phrases, case=False, na=False)]
print(f"\nComments matching mod/bot boilerplate phrases: {len(bot_text_hits)}")

ALL ASSERTIONS PASSED: 0 AutoModerator, 0 [deleted]/[removed], 0 subcomments, 0 duplicates, 100% paired threads.

=== TOP 10 COMMENT AUTHORS (ANONYMIZED BOT CHECK) ===
author_anon
user_cfa7403ce7    155
user_b32aa49756     74
user_3bd41656a6     65
user_c1330ea0ee     55
user_a971e3d34c     55
user_16a493e40f     53
user_633ca0eddb     52
user_c3e22ef10d     44
user_f52774c907     43
user_64b33de9de     42
Name: count, dtype: int64

Comments matching mod/bot boilerplate phrases: 0


## 4. Exploratory Data Analysis (EDA)

We examine the structure and distributions of the cleaned corpus to inform downstream annotation sampling, classification, and statistical modeling:
1. **Subreddit Distribution:** Post vs. comment volume across the four target subreddits.
2. **Temporal Coverage:** Yearly post counts (noting heavy concentration in 2024–2026 and sparser counts in earlier years).
3. **Thread Density & Text Length:** Comments per post and word-count distributions.
4. **Short Comments & Flair Coverage:** Frequency of ultra-short comments (`< 3` words) and non-null `postFlair` metadata.

In [12]:
posts["submittedAt"] = pd.to_datetime(posts["submittedAt"], utc=True)
comments["submittedAt"] = pd.to_datetime(comments["submittedAt"], utc=True)

posts["year"] = posts["submittedAt"].dt.year
posts["word_count"] = posts["bodyText"].str.split().str.len()
comments["word_count"] = comments["bodyText"].str.split().str.len()

print("=== 1. COUNTS BY SUBREDDIT ===")
eda_sub = pd.DataFrame({
    "clean_posts": posts["subreddit"].value_counts(),
    "clean_comments": comments["subreddit"].value_counts()
})
eda_sub["avg_comments_per_post"] = (eda_sub["clean_comments"] / eda_sub["clean_posts"]).round(1)
display(eda_sub)

print("\n=== 2. POSTS BY YEAR (TEMPORAL COVERAGE) ===")
display(posts["year"].value_counts().sort_index().to_frame("post_count"))

print("\n=== 3. COMMENTS PER POST & WORD LENGTH SUMMARY ===")
comments_per_post = comments.groupby("parentPostId").size()
print(f"Posts with >= 1 clean comment:  {len(comments_per_post):,} / {len(posts):,}")
print(f"Posts with >= 5 clean comments: {(comments_per_post >= 5).sum():,} / {len(posts):,}")
print(f"Posts with >= 10 clean comments:{(comments_per_post >= 10).sum():,} / {len(posts):,}")
print(f"Median clean comments per post: {comments_per_post.median():.1f} (mean: {comments_per_post.mean():.1f}, max: {comments_per_post.max()})")
print(f"Median post length: {posts['word_count'].median():.0f} words | Median comment length: {comments['word_count'].median():.0f} words")

print("\n=== 4. SHORT COMMENTS & FLAIR COVERAGE ===")
short_comments = (comments["word_count"] < 3).sum()
print(f"Comments with < 3 words: {short_comments:,} / {len(comments):,} ({short_comments/len(comments):.1%})")
print(f"Posts with non-null postFlair: {posts['postFlair'].notna().sum():,} / {len(posts):,} ({posts['postFlair'].notna().mean():.1%})")

=== 1. COUNTS BY SUBREDDIT ===


,clean_posts,clean_comments,avg_comments_per_post
subreddit,,,
r/dating_advice,335,5718,17.1
r/datingadvice,993,4565,4.6
r/relationship_advice,205,3343,16.3
r/relationshipadvice,808,6450,8.0



=== 2. POSTS BY YEAR (TEMPORAL COVERAGE) ===


,post_count
year,
2010,1
2011,10
2012,5
2013,11
2014,3
2018,3
2019,31
2020,77
2021,27



=== 3. COMMENTS PER POST & WORD LENGTH SUMMARY ===
Posts with >= 1 clean comment:  2,341 / 2,341
Posts with >= 5 clean comments: 1,394 / 2,341
Posts with >= 10 clean comments:889 / 2,341
Median clean comments per post: 6.0 (mean: 8.6, max: 31)
Median post length: 254 words | Median comment length: 41 words

=== 4. SHORT COMMENTS & FLAIR COVERAGE ===
Comments with < 3 words: 304 / 20,076 (1.5%)
Posts with non-null postFlair: 509 / 2,341 (21.7%)


## 5. Anonymize & Export Datasets

To protect user privacy and comply with Reddit's Terms of Service:
1. **Author Anonymization:** All Reddit usernames are replaced with a salted one-way SHA-256 hash (`author_anon`), preserving unique-author grouping without exposing personal handles.
2. **Full Private Datasets (`clean_data/`, excluded from public GitHub):**
   - `posts_clean.csv.gz` & `comments_clean.csv.gz`: Complete paired dataset (`2,341` posts and `20,076` top-level comments sorted by descending score) for problem classification and longitudinal modeling.
   - `posts_active_5plus.csv.gz` & `comments_active_5plus.csv.gz`: Active-threads subset (`>= 5` top-level comments per post) for multi-voice opinion summarization and sensitivity checks.
3. **Public GitHub Sample (`public_sample/`, committed to `data/sample/`):**
   - `posts_sample_anon.csv` & `comments_sample_anon.csv`: A reproducible 50-thread anonymized sample (`random_state=42`) allowing reviewers to inspect the exact schema and data structure without running code.

In [13]:
import hashlib

os.makedirs("clean_data", exist_ok=True)
os.makedirs("public_sample", exist_ok=True)

# 1. One-way salted hash for author usernames
SALT = "ta2_reddit_2026"
posts["author_anon"] = posts["author"].apply(
    lambda x: "user_" + hashlib.sha256(f"{SALT}_{x}".encode()).hexdigest()[:10]
)
comments["author_anon"] = comments["author"].apply(
    lambda x: "user_" + hashlib.sha256(f"{SALT}_{x}".encode()).hexdigest()[:10]
)

post_cols = [
    "submissionId", "subreddit", "submittedAt", "author_anon",
    "title", "bodyText", "postFlair", "score", "upvoteRatio", "numComments"
]
comment_cols = [
    "submissionId", "parentPostId", "subreddit", "submittedAt",
    "author_anon", "bodyText", "score", "depth"
]

# 2. Prepare full clean dataset (all posts with >= 1 clean top-level comment)
posts_out = posts[post_cols].reset_index(drop=True)
comments_out = (
    comments[comment_cols]
    .sort_values(["parentPostId", "score"], ascending=[True, False])
    .reset_index(drop=True)
)

# 3. Prepare active-threads subset (posts with >= 5 clean top-level comments)
comment_counts = comments_out.groupby("parentPostId").size()
active_post_ids = set(comment_counts[comment_counts >= 5].index)

posts_active = posts_out[posts_out["submissionId"].isin(active_post_ids)].reset_index(drop=True)
comments_active = comments_out[comments_out["parentPostId"].isin(active_post_ids)].reset_index(drop=True)

# 4. Export compressed private datasets for team use
private_exports = {
    "posts_clean.csv.gz": posts_out,
    "comments_clean.csv.gz": comments_out,
    "posts_active_5plus.csv.gz": posts_active,
    "comments_active_5plus.csv.gz": comments_active,
}

print("=== PRIVATE TEAM DATASETS (./clean_data/) ===")
for fname, df_export in private_exports.items():
    path = os.path.join("clean_data", fname)
    df_export.to_csv(path, index=False, compression="gzip")
    size_mb = os.path.getsize(path) / 1e6
    print(f"Saved {fname:<30} | {len(df_export):>7,} rows | {size_mb:.2f} MB")

# 5. Export small 50-thread anonymized sample for public GitHub repo
sample_posts = posts_active.sample(n=50, random_state=42).reset_index(drop=True)
sample_comments = comments_active[
    comments_active["parentPostId"].isin(set(sample_posts["submissionId"]))
].reset_index(drop=True)

sample_posts.to_csv("public_sample/posts_sample_anon.csv", index=False)
sample_comments.to_csv("public_sample/comments_sample_anon.csv", index=False)

print("\n=== PUBLIC GITHUB SAMPLE (./public_sample/) ===")
print(f"Saved posts_sample_anon.csv        | {len(sample_posts):>7,} rows")
print(f"Saved comments_sample_anon.csv     | {len(sample_comments):>7,} rows")

=== PRIVATE TEAM DATASETS (./clean_data/) ===
Saved posts_clean.csv.gz             |   2,341 rows | 1.75 MB
Saved comments_clean.csv.gz          |  20,076 rows | 3.47 MB
Saved posts_active_5plus.csv.gz      |   1,394 rows | 1.09 MB
Saved comments_active_5plus.csv.gz   |  17,930 rows | 3.06 MB

=== PUBLIC GITHUB SAMPLE (./public_sample/) ===
Saved posts_sample_anon.csv        |      50 rows
Saved comments_sample_anon.csv     |     684 rows


## 6. Post-Export Verification & Sample Thread Inspection

Finally, we reload all exported files directly from disk to confirm zero file corruption, verify 1-to-1 thread alignment across the full dataset, the `5+ comments` subset, and the public sample, and print an example thread for visual inspection.

In [14]:
# 1. Reload exported files from disk
posts_check = pd.read_csv("clean_data/posts_clean.csv.gz")
comments_check = pd.read_csv("clean_data/comments_clean.csv.gz")
posts_5plus_check = pd.read_csv("clean_data/posts_active_5plus.csv.gz")
comments_5plus_check = pd.read_csv("clean_data/comments_active_5plus.csv.gz")
sample_posts_check = pd.read_csv("public_sample/posts_sample_anon.csv")
sample_comments_check = pd.read_csv("public_sample/comments_sample_anon.csv")

# 2. Verify row counts, anonymization, and post-comment ID alignment
assert len(posts_check) == len(posts_out), "Row mismatch in posts_clean!"
assert len(comments_check) == len(comments_out), "Row mismatch in comments_clean!"
assert "author" not in posts_check.columns and "author_anon" in posts_check.columns, "Raw author column exposed!"
assert set(comments_check["parentPostId"]) == set(posts_check["submissionId"]), "ID mismatch in full dataset!"
assert set(comments_5plus_check["parentPostId"]) == set(posts_5plus_check["submissionId"]), "ID mismatch in 5+ subset!"
assert (comments_5plus_check.groupby("parentPostId").size() >= 5).all(), "Found thread with <5 comments in 5+ subset!"
assert set(sample_comments_check["parentPostId"]) == set(sample_posts_check["submissionId"]), "ID mismatch in public sample!"

print("=== ALL RELOAD & INTEGRITY CHECKS PASSED ===")
print(f"Full dataset:       {len(posts_check):,} posts | {len(comments_check):,} comments")
print(f"Active (5+) subset: {len(posts_5plus_check):,} posts | {len(comments_5plus_check):,} comments")
print(f"Public sample:      {len(sample_posts_check):,} posts | {len(sample_comments_check):,} comments\n")

# 3. Visual sanity check: inspect 1 sample post and its top 3 cleaned comments
sample_post = sample_posts_check.iloc[0]
sample_comments_preview = sample_comments_check[
    sample_comments_check["parentPostId"] == sample_post["submissionId"]
].head(3)

print("=" * 70)
print(f"SAMPLE THREAD [{sample_post['subreddit']}] | Post ID: {sample_post['submissionId']} | Score: {sample_post['score']}")
print(f"TITLE: {sample_post['title']}")
print(f"POST BODY: {sample_post['bodyText'][:350]}...")
print("-" * 70)
for _, row in sample_comments_preview.iterrows():
    print(f"  -> Comment (score={row['score']}, {row['author_anon']}): {row['bodyText'][:250]}...")

=== ALL RELOAD & INTEGRITY CHECKS PASSED ===
Full dataset:       2,341 posts | 20,076 comments
Active (5+) subset: 1,394 posts | 17,930 comments
Public sample:      50 posts | 684 comments

SAMPLE THREAD [r/relationship_advice] | Post ID: 1x8tsj7 | Score: 669
TITLE: I (37M) ended things with girlfriend (38F) over long term incompatibilities
POST BODY: I (37M) had been dating my girlfriend E (38F) for a little over a year now. We absolutely adore each other. She genuinely makes my life better in ways I didn't know I needed, and I think/hope I do the same for her.
    
      At the beginning of the relationship E had stated she didn't want kids. I did, but I am nearing 40 and I have come to accept...
----------------------------------------------------------------------
  -> Comment (score=1307, user_a6a0be40c4): I’m going to be brutal here. Honestly, you should have stopped this when she said she didn’t want kids. Now, more emotions, and hence more confusion and hurt is involved. You ca